In [5]:
import pandas as pd

# Roll number
roll_number = "1024170060"

# Last two digits
last_two_digits = roll_number[-2:]

# Category mapping
categories = ["billing", "account", "general"]

# Fixed FAQ entries
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

# Personalized entries for digits 6 and 0
personalized_entries = []

for digit in last_two_digits:
    d = int(digit)
    category = categories[d % 3]

    if category == "billing":
        question = "how can i check my payment status"
        answer = "You can check your payment status from the Billing section."
        keywords = "payment status billing transaction"

    elif category == "account":
        question = "how do i update my registered mobile number"
        answer = "Go to Account Settings and update your registered mobile number."
        keywords = "mobile number update account"

    else:
        question = "where can i find general information"
        answer = "General information is available in the Help section."
        keywords = "information help details"

    personalized_entries.append({
        "question": question,
        "answer": answer,
        "keywords": keywords,
        "category": category
    })

# Create final 6-row DataFrame
df = pd.DataFrame(fixed_entries + personalized_entries)

print("Final FAQ Knowledge Base:")
print(df.to_string(index=False))

Final FAQ Knowledge Base:
                         question                                                      answer                           keywords category
           what is the annual fee                                   The annual fee is Rs 500.              fee cost price charge  billing
            how to reset password                            Go to Settings > Reset Password.               password reset login  account
      what are your working hours                                   We are open 9 AM to 5 PM.             hours timing open time  general
            how can i pay the fee                  You can pay via UPI, card, or net banking.                pay payment upi fee  billing
how can i check my payment status You can check your payment status from the Billing section. payment status billing transaction  billing
how can i check my payment status You can check your payment status from the Billing section. payment status billing transaction  billing


In [6]:
def score_query(query, df):
    query_words = set(query.lower().split())
    results = []

    for _, row in df.iterrows():
        text = (row["question"] + " " + row["keywords"]).lower()
        text_words = set(text.split())

        score = len(query_words.intersection(text_words))

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": score
            })

    results = sorted(
        results,
        key=lambda x: x["confidence"],
        reverse=True
    )

    return pd.DataFrame(results)


query = input("Enter your question: ")

results = score_query(query, df)

print("\nMatching FAQ entries:")

if results.empty:
    print("No matching FAQ entry found.")
else:
    print(results.to_string(index=False))


Matching FAQ entries:
                   question                                     answer category  confidence
     what is the annual fee                  The annual fee is Rs 500.  billing           4
what are your working hours                  We are open 9 AM to 5 PM.  general           1
      how can i pay the fee You can pay via UPI, card, or net banking.  billing           1


In [7]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]


# Category of the first personalized entry
personalized_category = personalized_entries[0]["category"]

print("Selected category:", personalized_category)
print("\nQuestions belonging to this category:")

category_results = same_category(personalized_category, df)

print(category_results.to_string(index=False))

Selected category: billing

Questions belonging to this category:
                         question                                                      answer                           keywords category
           what is the annual fee                                   The annual fee is Rs 500.              fee cost price charge  billing
            how can i pay the fee                  You can pay via UPI, card, or net banking.                pay payment upi fee  billing
how can i check my payment status You can check your payment status from the Billing section. payment status billing transaction  billing
how can i check my payment status You can check your payment status from the Billing section. payment status billing transaction  billing


In [8]:
# Display all entries
print("Current FAQ Knowledge Base:\n")
print(df[["question", "keywords", "category"]].to_string())

# Select an entry
entry_index = int(input("\nEnter the index of the entry you want to update: "))

# Get new keyword
new_keyword = input("Enter the new keyword: ").strip().lower()

# Add new keyword
df.loc[entry_index, "keywords"] = (
    df.loc[entry_index, "keywords"] + " " + new_keyword
)

# Save updated DataFrame
filename = roll_number + "_faq_data.csv"
df.to_csv(filename, index=False)

print("\nUpdated entry:")
print(df.loc[entry_index].to_string())

print("\nUpdated DataFrame saved as:", filename)

Current FAQ Knowledge Base:

                            question                            keywords category
0             what is the annual fee               fee cost price charge  billing
1              how to reset password                password reset login  account
2        what are your working hours              hours timing open time  general
3              how can i pay the fee                 pay payment upi fee  billing
4  how can i check my payment status  payment status billing transaction  billing
5  how can i check my payment status  payment status billing transaction  billing

Updated entry:
question                    how can i check my payment status
answer      You can check your payment status from the Bil...
keywords           payment status billing transaction finance
category                                              billing

Updated DataFrame saved as: 1024170060_faq_data.csv


In [9]:
category_counts = df.groupby("category").size().reset_index(name="count")

print("Number of FAQ entries per category:")
print(category_counts.to_string(index=False))

Number of FAQ entries per category:
category  count
 account      1
 billing      4
 general      1


In [10]:
def score_query_with_ties(query, df):
    query_words = set(query.lower().split())
    results = []

    for _, row in df.iterrows():
        text = (row["question"] + " " + row["keywords"]).lower()
        text_words = set(text.split())

        score = len(query_words.intersection(text_words))

        if score > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "confidence": score
            })

    if not results:
        print("No matching FAQ entry found.")
        return

    # Sort by confidence
    results = sorted(
        results,
        key=lambda x: x["confidence"],
        reverse=True
    )

    # Highest score
    highest_score = results[0]["confidence"]

    # Keep all entries with highest score
    best_matches = [
        result for result in results
        if result["confidence"] == highest_score
    ]

    print("\nBest matching FAQ entries:")
    print("Highest confidence score:", highest_score)

    for result in best_matches:
        print("\nQuestion:", result["question"])
        print("Answer:", result["answer"])
        print("Category:", result["category"])
        print("Confidence:", result["confidence"])

    if len(best_matches) > 1:
        print("\nTie detected: multiple entries have the same highest score.")
    else:
        print("\nNo tie: only one entry has the highest score.")


# Tie demonstration
print("========== TIE DEMONSTRATION ==========")
score_query_with_ties("fee", df)


# Non-tie demonstration
print("\n\n========== NON-TIE DEMONSTRATION ==========")
score_query_with_ties("password", df)

========== TIE DEMONSTRATION ==========

Best matching FAQ entries:
Highest confidence score: 1

Question: what is the annual fee
Answer: The annual fee is Rs 500.
Category: billing
Confidence: 1

Question: how can i pay the fee
Answer: You can pay via UPI, card, or net banking.
Category: billing
Confidence: 1

Tie detected: multiple entries have the same highest score.


========== NON-TIE DEMONSTRATION ==========

Best matching FAQ entries:
Highest confidence score: 1

Question: how to reset password
Answer: Go to Settings > Reset Password.
Category: account
Confidence: 1

No tie: only one entry has the highest score.
